# Introduction

In [31]:
import numpy as np

# Problem 1: Generating Random Boolean Functions
The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

In [32]:
import random
from itertools import product

def random_constant_balanced(rng=None):
    #if no random generator is provided, use the standard one
    if rng is None:
        rng = random
    
    #decide whether to create a constant or balanced function
    choice = rng.choice(["constant", "balanced"])

    if choice == "constant":
        #pick a single output value (True or False)
        value = rng.choice([False, True])
        
        #define a function that always returns that value, ignoring inputs
        def constant_function(a, b, c, d):
            return value
            
        return constant_function

    if choice == "balanced":
        #generate all 16 possible combinations of 4 inputs
        all_possible_inputs = list(product([False, True], repeat=4))
        
        #randomly pick exactly 8 of them to return true
        inputs_that_return_true = rng.sample(all_possible_inputs, k=8)
        
        #define a function that checks if the input is in our chosen list
        def balanced_function(a, b, c, d):
            current_input = (a, b, c, d)
            if current_input in inputs_that_return_true:
                return True
            else:
                return False
                
        return balanced_function

In [33]:
def classify_constant_or_balanced(f):
    #create a list of all 16 inputs to test the function
    inputs = list(product([False, True], repeat=4))
    
    #evaluate the function for every input and store the results
    outputs = []
    for inp in inputs:
        result = f(*inp)
        outputs.append(result)
    
    #count how many times the function returned true
    true_count = sum(outputs)

    #constant if all outputs are the same (all 0 or all 16 are true)
    if true_count == 0 or true_count == 16:
        return "constant"
        
    #balanced if exactly half (8) are true
    if true_count == 8:
        return "balanced"
        
    return "neither"

In [34]:
#test cases
def test_constant_true(a, b, c, d):
    #always returns true (constant)
    return True


def test_constant_false(a, b, c, d):
    #always returns false (constant)
    return False


def test_balanced_parity(a, b, c, d):
    #even parity gives a balanced function over 4 bits
    return (a + b + c + d) % 2 == 0


#run tests
assert classify_constant_or_balanced(test_constant_true) == "constant"
print("test_constant_true: passed")
assert classify_constant_or_balanced(test_constant_false) == "constant"
print("test_constant_false: passed")
assert classify_constant_or_balanced(test_balanced_parity) == "balanced"
print("test_balanced_parity: passed")

test_constant_true: passed
test_constant_false: passed
test_balanced_parity: passed


In [35]:
def format_truth_table(f):
    lines = []
    for inp in product([0, 1], repeat=4):
        #convert 0/1 integers to boolean False/True for the function
        a = bool(inp[0])
        b = bool(inp[1])
        c = bool(inp[2])
        d = bool(inp[3])
        
        #get the output from the function
        result = f(a, b, c, d)
        
        #convert the result back to 0 or 1 for printing
        if result == True:
            out = 1
        else:
            out = 0
            
        lines.append(f"{inp} -> {out}")
    return lines

In [ ]:
#random trials should always be constant or balanced by construction
for _ in range(5):
    f = random_constant_balanced()
    assert classify_constant_or_balanced(f) in {"constant", "balanced"}

#seeded rng makes the test deterministic
#use a specific seed so we get the same function every time we run this
seeded_rng = random.Random(123)
f_seeded = random_constant_balanced(seeded_rng)

#check if the function is valid using our helper
result = classify_constant_or_balanced(f_seeded)

if result == "constant" or result == "balanced":
    print(f"seeded test passed: generated a {result} function.")
else:
    print("seeded test failed.")

#results and demonstration: truth tables
for i in range(1, 3):
    f = random_constant_balanced()
    label = classify_constant_or_balanced(f)
    print("Truth Table:")
    print(label)
    print(f"Try {i}:")
    for line in format_truth_table(f):
        print(line)
    print()

seeded test passed: generated a constant function.
Truth Table:
constant
Try 1:
(0, 0, 0, 0) -> 1
(0, 0, 0, 1) -> 1
(0, 0, 1, 0) -> 1
(0, 0, 1, 1) -> 1
(0, 1, 0, 0) -> 1
(0, 1, 0, 1) -> 1
(0, 1, 1, 0) -> 1
(0, 1, 1, 1) -> 1
(1, 0, 0, 0) -> 1
(1, 0, 0, 1) -> 1
(1, 0, 1, 0) -> 1
(1, 0, 1, 1) -> 1
(1, 1, 0, 0) -> 1
(1, 1, 0, 1) -> 1
(1, 1, 1, 0) -> 1
(1, 1, 1, 1) -> 1

Truth Table:
balanced
Try 2:
(0, 0, 0, 0) -> 1
(0, 0, 0, 1) -> 1
(0, 0, 1, 0) -> 1
(0, 0, 1, 1) -> 0
(0, 1, 0, 0) -> 1
(0, 1, 0, 1) -> 0
(0, 1, 1, 0) -> 0
(0, 1, 1, 1) -> 1
(1, 0, 0, 0) -> 1
(1, 0, 0, 1) -> 1
(1, 0, 1, 0) -> 0
(1, 0, 1, 1) -> 0
(1, 1, 0, 0) -> 0
(1, 1, 0, 1) -> 1
(1, 1, 1, 0) -> 0
(1, 1, 1, 1) -> 0



# Problem 2: Classical Testing for Function Type
Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

# Problem 3: Quantum Oracles
Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

# Problem 4: Deutsch's Algorithm with Qiskit
Use Qiskit to design a quantum circuit that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.



# Problem 5: Scaling to the Deutsch–Jozsa Algorithm
The Deutsch–Jozsa algorithm generalizes Deutsch's approach to functions with several input bits. Use Qiskit to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.